# Urban Planning & Infrastructure Monitoring: SAR Subsidence + Urban Growth
**PyGeoVision v2.0 — SAR Applications Notebook 06**

| Timescale | Sensor | What we monitor |
|---|---|---|
| Daily–monthly | S2 optical + ChangeFormer | Construction progress, new buildings |
| mm/year | S1 SAR temporal | Ground subsidence, building settling |

**Study area:** Jakarta, Indonesia (sinking up to 25 cm/year) + Dubai, UAE  
**Models:** DInSAR proxy · SegFormer+DINOv3 building extraction · ChangeFormer urban growth


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  PyGeoVision v2.0 — Notebook Setup Cell                                    ║
# ║  Run this cell FIRST. Defines all imports and helper functions.             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import warnings; warnings.filterwarnings('ignore')
import json
import pathlib

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

# ── PyTorch ────────────────────────────────────────────────────────────────────
import torch
from torch.utils.data import DataLoader, TensorDataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Rasterio ───────────────────────────────────────────────────────────────────
import rasterio

# ── PyGeoVision — core client ─────────────────────────────────────────────────
import pygeovision as pgv
from pygeovision import PyGeoVision

client = PyGeoVision()

# ── PyGeoVision — models ──────────────────────────────────────────────────────
from pygeospace.folium import Map as PgsMap
from pygeospace.folium import RasterLayer, VectorLayer

# ── PyGeoVision — training ────────────────────────────────────────────────────
from pygeovision.datasets.analysis import DatasetAnalyzer
from pygeovision.datasets.loader import DatasetLoader

# ── PyGeoVision — data & datasets ────────────────────────────────────────────
from pygeovision.datasets.registry import DatasetRegistry
from pygeovision.inference.tiled import TiledInference
from pygeovision.models import get_model
from pygeovision.models.change_detection.changeformer import ChangeFormer  # NOT ChangeDetection
from pygeovision.models.foundation.dinov3 import DINOv3Backbone, get_transform
from pygeovision.training.checkpoint import CheckpointManager
from pygeovision.training.mixed_precision import MixedPrecisionManager

reg    = DatasetRegistry()
loader = DatasetLoader()

# ── PyGeoSpace — interactive maps (graceful fallback if not installed) ─────────
try:
# PyGeoSpace helpers: show_raster_map, show_split_map, show_vector_map (defined in Setup cell)
    HAS_PYGEOSPACE = True
    print("PyGeoSpace: ✓ interactive maps enabled")
except ImportError:
    HAS_PYGEOSPACE = False
    print("PyGeoSpace: not installed — using matplotlib fallback")
    print("  → pip install pygeospace   to enable interactive folium maps")

# ── Map helper functions (work with or without PyGeoSpace) ────────────────────
def show_raster_map(arr, bounds, title="", cmap='tab20', vmin=None, vmax=None,
                    save_path=None, colorbar_label="", legend_dict=None):
    """Raster map — PyGeoSpace (interactive) or matplotlib (static fallback)."""
    if HAS_PYGEOSPACE:
        lat_c=(bounds[1]+bounds[3])/2; lon_c=(bounds[0]+bounds[2])/2
        m = PgsMap(location=(lat_c,lon_c), zoom_start=11,
                   basemap='esri-satellite', title=title)
        m.add_layer(RasterLayer(arr.astype('float32'),
                                bounds=(bounds[0],bounds[1],bounds[2],bounds[3]),
                                cmap=cmap, opacity=0.75,
                                vmin=vmin or float(arr.min()),
                                vmax=vmax or float(arr.max())))
        if colorbar_label:
            m.add_colorbar(vmin or float(arr.min()), vmax or float(arr.max()),
                           cmap=cmap, caption=colorbar_label)
        if legend_dict: m.add_legend(legend_dict)
        m.add_title(title).add_scale()
        if save_path: m.save(str(save_path))
        m.show()
        return m
    else:
        fig, ax = plt.subplots(figsize=(10,8))
        im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title, fontsize=14, pad=10); ax.axis('off')
        plt.colorbar(im, ax=ax, label=colorbar_label or cmap, fraction=0.03)
        if legend_dict:
            patches = [mpatches.Patch(color=c, label=n) for n,c in legend_dict.items()]
            ax.legend(handles=patches, loc='lower right', fontsize=9,
                      framealpha=0.85, borderpad=0.6)
        plt.tight_layout()
        if save_path:
            p = str(save_path).replace('.html','.png')
            plt.savefig(p, dpi=150, bbox_inches='tight'); print(f"Saved: {p}")
        plt.show(); return fig

def show_split_map(arr_l, arr_r, bounds, title="",
                   left_label="Pre-trained", right_label="Fine-tuned",
                   cmap='tab20', save_path=None, legend_dict=None):
    """Split comparison — PyGeoSpace drag-divider or side-by-side matplotlib."""
    if HAS_PYGEOSPACE:
        lat_c=(bounds[1]+bounds[3])/2; lon_c=(bounds[0]+bounds[2])/2
        m = PgsMap(location=(lat_c,lon_c), zoom_start=11, basemap='esri-satellite')
        m.split_map(
            left_layer =RasterLayer(arr_l.astype('float32'),
                                    bounds=(bounds[0],bounds[1],bounds[2],bounds[3]),
                                    cmap=cmap, opacity=0.82),
            right_layer=RasterLayer(arr_r.astype('float32'),
                                    bounds=(bounds[0],bounds[1],bounds[2],bounds[3]),
                                    cmap=cmap, opacity=0.82),
            left_label=left_label, right_label=right_label)
        m.add_title(title, subtitle="Drag divider ← → to compare")
        if legend_dict: m.add_legend(legend_dict)
        m.add_full_ui()
        if save_path: m.save(str(save_path))
        m.show(); return m
    else:
        fig, axes = plt.subplots(1,2,figsize=(16,7), constrained_layout=True)
        axes[0].imshow(arr_l, cmap=cmap); axes[0].set_title(left_label, fontsize=13); axes[0].axis('off')
        axes[1].imshow(arr_r, cmap=cmap); axes[1].set_title(right_label,fontsize=13); axes[1].axis('off')
        fig.suptitle(title, fontsize=15)
        if save_path:
            p = str(save_path).replace('.html','.png')
            plt.savefig(p, dpi=150, bbox_inches='tight'); print(f"Saved: {p}")
        plt.show(); return fig

def show_vector_map(vector_path, raster_arr=None, bounds=None, title="",
                    fill_color='#ff4400', save_path=None):
    """Vector polygon map with optional raster background."""
    if HAS_PYGEOSPACE and bounds:
        lat_c=(bounds[1]+bounds[3])/2; lon_c=(bounds[0]+bounds[2])/2
        m = PgsMap(location=(lat_c,lon_c), zoom_start=11,
                   basemap='esri-satellite', title=title)
        if raster_arr is not None:
            m.add_layer(RasterLayer(raster_arr.astype('float32'),
                                    bounds=(bounds[0],bounds[1],bounds[2],bounds[3]),
                                    cmap='tab20', opacity=0.55))
        if pathlib.Path(str(vector_path)).exists():
            m.add_layer(VectorLayer(str(vector_path),
                                    fill_color=fill_color, fill_opacity=0.45, weight=1.5))
        m.add_title(title).add_scale()
        if save_path: m.save(str(save_path))
        m.show(); return m
    else:
        print(f"[map] {title}")
        if pathlib.Path(str(vector_path)).exists():
            with open(vector_path) as f: gj = json.load(f)
            print(f"  {len(gj.get('features',[]))} features in {pathlib.Path(vector_path).name}")
        return None

# ── Training helper functions ─────────────────────────────────────────────────
def chips_from_raster(path, chip_size=256, max_chips=120):
    """Slice a preprocessed raster into (N, C, chip_size, chip_size) chips."""
    with rasterio.open(str(path)) as src:
        data = src.read().astype('float32')
        bounds = src.bounds
    C, H, W = data.shape
    chips = []
    for y in range(0, H - chip_size + 1, chip_size):
        for x in range(0, W - chip_size + 1, chip_size):
            chip = data[:, y:y+chip_size, x:x+chip_size]
            if not np.isnan(chip).all():
                chips.append(chip)
            if len(chips) >= max_chips:
                return np.stack(chips), bounds
    if chips:
        return np.stack(chips), bounds
    return np.zeros((1, C, chip_size, chip_size), 'float32'), bounds

def aug_chip(chip):
    """Random flip + rotate + noise augmentation for geospatial chips."""
    if np.random.rand() > 0.5: chip = chip[:, ::-1, :].copy()
    if np.random.rand() > 0.5: chip = chip[:, :, ::-1].copy()
    k = np.random.randint(4)
    if k: chip = np.rot90(chip, k, axes=(1,2)).copy()
    chip = np.clip(chip + np.random.normal(0, 0.005, chip.shape).astype('float32'), 0, 1)
    return chip

def make_loaders(chips, labels, batch_size=6, split=0.80):
    """Build train/val DataLoaders with augmentation on the training split."""
    n   = len(chips)
    idx = np.random.permutation(n)
    s   = int(n * split)
    tr_chips = np.stack([aug_chip(chips[i]) for i in idx[:s]])
    va_chips  = chips[idx[s:]]
    tr = TensorDataset(torch.tensor(tr_chips),   torch.tensor(labels[idx[:s]]))
    va = TensorDataset(torch.tensor(va_chips),    torch.tensor(labels[idx[s:]]))
    return (DataLoader(tr, batch_size=batch_size, shuffle=True,  num_workers=0),
            DataLoader(va, batch_size=batch_size, shuffle=False, num_workers=0))

def run_epochs(model, tr_loader, va_loader, n_epochs, criterion,
               lr=1e-4, n_classes=2, out_dir='./ckpt', label='model'):
    """Standard training loop with cosine LR, mixed precision, and checkpointing."""
    out_dir = pathlib.Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    opt   = torch.optim.AdamW(filter(lambda p:p.requires_grad, model.parameters()),
                               lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=lr*0.01)
    mp    = MixedPrecisionManager(precision='auto', enabled=(DEVICE=='cuda'))
    ckpt  = CheckpointManager(str(out_dir), monitor='val_miou', mode='max')
    best  = 0.0
    for ep in range(n_epochs):
        model.train(); ep_loss = []
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            with mp.autocast():
                logits = model(xb)
                loss   = criterion(logits, yb)
            mp.scale_and_step(loss, opt)
            ep_loss.append(loss.item())
        model.eval(); vp_all, vl_all = [], []
        with torch.no_grad():
            for xb, yb in va_loader:
                logits = model(xb.to(DEVICE))
                vp_all.append(logits.argmax(1).cpu())
                vl_all.append(yb)
        vp = torch.cat(vp_all).numpy(); vl = torch.cat(vl_all).numpy()
        ious = []
        for c in range(n_classes):
            tp = ((vp==c)&(vl==c)).sum(); fp = ((vp==c)&(vl!=c)).sum(); fn = ((vp!=c)&(vl==c)).sum()
            ious.append(float(tp/(tp+fp+fn+1e-8)))
        miou = float(np.mean(ious))
        sched.step()
        if miou > best:
            best = miou
            ckpt.save(ep, model, opt, metrics={'val_miou': miou})
        if (ep+1) % max(1, n_epochs//5) == 0:
            print(f"  Ep {ep+1:3d}/{n_epochs}  loss={np.mean(ep_loss):.4f}"
                  f"  val_mIoU={miou*100:.1f}%  best={best*100:.1f}%")
    return best, ckpt

def run_epochs_bitemporal(model, tr_loader, va_loader, n_epochs, criterion,
                           lr=1e-4, n_classes=2, out_dir='./ckpt', label='model'):
    """Training loop for bi-temporal models (ChangeFormer: takes 2 inputs)."""
    out_dir = pathlib.Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    ckpt  = CheckpointManager(str(out_dir), monitor='val_miou', mode='max')
    best  = 0.0
    n_bands_per = None   # inferred on first batch
    for ep in range(n_epochs):
        model.train(); ep_loss = []
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if n_bands_per is None: n_bands_per = xb.shape[1] // 2
            opt.zero_grad()
            logits = model(xb[:, :n_bands_per, ...], xb[:, n_bands_per:, ...])
            loss   = criterion(logits, yb)
            loss.backward(); opt.step()
            ep_loss.append(loss.item())
        model.eval(); vp_all, vl_all = [], []
        with torch.no_grad():
            for xb, yb in va_loader:
                logits = model(xb[:,:n_bands_per,...].to(DEVICE),
                               xb[:,n_bands_per:,...].to(DEVICE))
                vp_all.append(logits.argmax(1).cpu()); vl_all.append(yb)
        vp = torch.cat(vp_all).numpy(); vl = torch.cat(vl_all).numpy()
        ious = [float(((vp==c)&(vl==c)).sum()/(((vp==c)|(vl==c)).sum()+1e-8)) for c in range(n_classes)]
        miou = float(np.mean(ious)); sched.step()
        if miou > best:
            best = miou; ckpt.save(ep, model, opt, metrics={'val_miou': miou})
        if (ep+1) % max(1, n_epochs//5) == 0:
            print(f"  Ep {ep+1:3d}/{n_epochs}  loss={np.mean(ep_loss):.4f}"
                  f"  val_mIoU={miou*100:.1f}%  best={best*100:.1f}%")
    return best, ckpt

def compute_miou(pred, labels, n_classes):
    """Compute mean IoU across n_classes from flat numpy arrays."""
    ious = []
    for c in range(n_classes):
        tp = ((pred==c)&(labels==c)).sum()
        fp = ((pred==c)&(labels!=c)).sum()
        fn = ((pred!=c)&(labels==c)).sum()
        ious.append(float(tp/(tp+fp+fn+1e-8)))
    return float(np.mean(ious)), ious

def compare3(pred_zero, pred_bench, pred_ft, labels, n_classes, class_names, title=""):
    """Print 3-tier comparison table: zero-shot vs benchmark-trained vs domain fine-tuned."""
    m0, i0 = compute_miou(pred_zero,  labels.ravel() if labels.ndim>1 else labels, n_classes)
    mb, ib = compute_miou(pred_bench, labels.ravel() if labels.ndim>1 else labels, n_classes)
    mf, if_= compute_miou(pred_ft,    labels.ravel() if labels.ndim>1 else labels, n_classes)
    sep = '═' * 58
    print(f"\n  {sep}")
    print(f"  3-Tier Comparison{(' — '+title) if title else ''}")
    print(f"  {sep}")
    print(f"  {'Model':<30} {'mIoU':>8} {'Δ vs zero-shot':>16}")
    print(f"  {'─'*56}")
    print(f"  {'0. Zero-shot (pre-trained)':<30} {m0*100:>7.1f}%")
    print(f"  {'1. Benchmark-trained':<30} {mb*100:>7.1f}% {(mb-m0)*100:>+14.1f}pp")
    print(f"  {'2. Domain fine-tuned':<30} {mf*100:>7.1f}% {(mf-m0)*100:>+14.1f}pp")
    print(f"  {sep}")
    print(f"\n  {'Cls':<4} {'Name':<18} {'Zero':>7} {'Bench':>7} {'FT':>7}")
    print(f"  {'─'*48}")
    for i, name in enumerate(class_names[:n_classes]):
        print(f"  {i:<4} {name:<18} {i0[i]*100:>6.1f}% {ib[i]*100:>6.1f}% {if_[i]*100:>6.1f}%")
    return {'zero': m0, 'bench': mb, 'ft': mf,
            'per_class': {'zero':i0,'bench':ib,'ft':if_}}


# ── PyGeoVision — SAR bug-fix preprocessing modules (Jul 2026) ──────────────
from pygeovision.data.processors.sar import (
    clip_sar_to_bbox,
    despeckle_sar,
    linear_to_db,
    normalise_sar_for_ai,
    validate_sar_georeference,
)
from pygeovision.data.validators.georeference import (
    check_download_complete,
)


def prep_sar(raw_path, output_path, label, bbox_wgs84, bands,
             work_dir=None, filter_type='enhanced_lee', window_size=7):
    """
    Correct Sentinel-1 GRD preprocessing.
    Order: verify(S0) → validate_georef(S1) → despeckle_LINEAR(S6) → dB(S7) → norm(S8) → clip(S9)
    BUG 1 fix: validate_sar_georeference() detects/repairs identity-transform corruption
    BUG 2 fix: check_download_complete() catches partial downloads before despeckle
    BUG 3 fix: clip_sar_to_bbox() reprojects WGS84 bbox to raster CRS before clipping
    """
    if raw_path is None: print(f"  SKIP {label}: no path"); return None
    raw_path = pathlib.Path(str(raw_path))
    if not raw_path.exists(): print(f"  SKIP {label}: {raw_path.name} not found"); return None
    output_path = pathlib.Path(str(output_path))
    wdir = pathlib.Path(str(work_dir)) if work_dir else output_path.parent
    wdir.mkdir(parents=True, exist_ok=True)

    # S0: download completeness (BUG 2)
    dlc = check_download_complete(str(raw_path))
    if not dlc['complete']:
        print(f"  WARN {label}: incomplete download ({dlc['readable_tiles']}/{dlc['total_tiles']} tiles)")
        if dlc['readable_tiles'] == 0:
            print(f"  FAIL {label}: zero readable tiles — skip"); return None

    # S1: georeference validation (BUG 1)
    geo = validate_sar_georeference(str(raw_path))
    working = geo.repaired_path or str(raw_path)
    if not geo.valid:
        print(f"  WARN {label}: georeference invalid — {geo.errors}"); return None

    stem = raw_path.stem
    # S6: despeckle on LINEAR data — MUST be BEFORE dB conversion
    desp = str(wdir / f"{stem}_despeckle.tif")
    try:
        despeckle_sar(working, desp, filter_type=filter_type, window_size=window_size)
    except Exception as e:
        print(f"  WARN {label}: despeckle fallback ({e})")
        import shutil; shutil.copy2(working, desp)

    # S7: convert to dB (AFTER despeckle)
    db = str(wdir / f"{stem}_db.tif")
    linear_to_db(desp, db, clip_db_min=-35.0, clip_db_max=5.0)

    # S8: normalise [0,1] with fixed physical bounds
    norm = str(wdir / f"{stem}_norm.tif")
    normalise_sar_for_ai(db, norm, method='minmax_db')

    # S9: CRS-aware clip (BUG 3) — auto-reprojects WGS84 bbox to raster CRS
    try:
        clip_sar_to_bbox(norm, str(output_path), bbox_wgs84)
    except ValueError as e:
        print(f"  WARN {label}: clip failed ({e}); using full-extent output")
        import shutil; shutil.copy2(norm, str(output_path))

    with rasterio.open(str(output_path)) as src: arr = src.read()
    print(f"  ✓ {label}: {arr.shape}  [{arr.min():.4f},{arr.max():.4f}]  → {output_path.name}")
    return output_path


print("Setup complete:")
print(f"  Device        : {DEVICE}")
print(f"  PyGeoVision   : {pgv.__version__}")
print(f"  Registry      : {len(reg)} datasets")
print(f"  PyGeoSpace    : {'✓' if HAS_PYGEOSPACE else 'fallback (pip install pygeospace)'}")
print("  ChangeFormer  : ChangeFormer(num_classes=N, in_channels=C)")
print("  DiceFocalLoss : from pygeovision.ai.training.losses")


In [ ]:
# (removed orphaned: SENTINEL2_CANONICAL, HLS_SCALE_FACTOR))
print(client)

In [ ]:
# Jakarta subsidence monitoring
JAKARTA_BBOX = (106.80, -6.30, 107.00, -6.10)
DATE_EARLY   = ('2021-01-01', '2021-06-30')
DATE_LATE    = ('2024-01-01', '2024-06-30')
PROVIDERS    = ['planetary_computer']
DATA_DIR     = pathlib.Path('./results/sar06_urban')
DATA_DIR.mkdir(parents=True, exist_ok=True)
S1_BANDS = ['vv','vh']; S2_BANDS = ['B02','B03','B04','B08','B11','B12']
for d in ['s1_early','s1_late','s2_early','s2_late']: (DATA_DIR/d).mkdir(exist_ok=True)
s1_early=client.search(bbox=JAKARTA_BBOX, date_range=DATE_EARLY, providers=PROVIDERS, satellites=['Sentinel-1'], cloud_cover_max=100)
s1_late =client.search(bbox=JAKARTA_BBOX, date_range=DATE_LATE,  providers=PROVIDERS, satellites=['Sentinel-1'], cloud_cover_max=100)
s2_early=client.search(bbox=JAKARTA_BBOX, date_range=DATE_EARLY, providers=PROVIDERS, cloud_cover_max=25)
s2_late =client.search(bbox=JAKARTA_BBOX, date_range=DATE_LATE,  providers=PROVIDERS, cloud_cover_max=25)
print(f"S1: {len(s1_early)} early / {len(s1_late)} late  S2: {len(s2_early)} early / {len(s2_late)} late")

In [ ]:
# Download
dl_s1e=client.download(s1_early[:1],output_dir=str(DATA_DIR/'s1_early'),bands=S1_BANDS,post_process=['reproject:EPSG:32748','cog'])
dl_s1l=client.download(s1_late[:1], output_dir=str(DATA_DIR/'s1_late'), bands=S1_BANDS,post_process=['reproject:EPSG:32748','cog'])
dl_s2e=client.download([s2_early[0]],output_dir=str(DATA_DIR/'s2_early'),bands=S2_BANDS,post_process=['reproject:EPSG:32748','cog'])
dl_s2l=client.download([s2_late[0]], output_dir=str(DATA_DIR/'s2_late'), bands=S2_BANDS,post_process=['reproject:EPSG:32748','cog'])
s1ep=dl_s1e[0].path if dl_s1e and dl_s1e[0].success else None
s1lp=dl_s1l[0].path if dl_s1l and dl_s1l[0].success else None
s2ep=dl_s2e[0].path if dl_s2e and dl_s2e[0].success else None
s2lp=dl_s2l[0].path if dl_s2l and dl_s2l[0].success else None

In [ ]:
# Preprocessing (identical SAR parameters)
S1ER=DATA_DIR/'s1_early_ready.tif'; S1LR=DATA_DIR/'s1_late_ready.tif'
S2ER=DATA_DIR/'s2_early_ready.tif'; S2LR=DATA_DIR/'s2_late_ready.tif'
for path,out,lbl,sar in [(s1ep,S1ER,'S1-early',True),(s1lp,S1LR,'S1-late',True),
                          (s2ep,S2ER,'S2-early',False),(s2lp,S2LR,'S2-late',False)]:
    if not path: continue
    if sar:
        # SAR despeckle now handled by prep_sar() — see correct order in setup cell
    # desp = desp=despeckle_sar(str(path, filter_type='enhanced_lee', window_size=7)  # REMOVED: private API,filter='enhanced_lee',window=5)
        r=client.prepare_for_ai(desp,stack_bands=S1_BANDS,bbox=JAKARTA_BBOX,normalise='minmax',model_type='segmentation',output_path=str(out))
    else:
        scl=next(iter(path.parent.rglob('*SCL*.tif')),None)
        r=client.prepare_for_ai(str(path),stack_bands=S2_BANDS,bbox=JAKARTA_BBOX,scl_path=str(scl) if scl else None,normalise='scale_factor',scale_factor=10000.0,model_type='segmentation',output_path=str(out))
    print(f"  {lbl}: shape={r['shape']}")

In [ ]:
# SAR subsidence proxy (backscatter change in urban areas)
SUBSIDENCE = DATA_DIR/'subsidence_proxy.tif'
if S1ER.exists() and S1LR.exists():
    with rasterio.open(str(S1ER)) as s1,rasterio.open(str(S1LR)) as s2:
        vv_e=s1.read(1).astype('float32'); vv_l=s2.read(1).astype('float32')
        prof=s1.profile.copy(); res_m=abs(s1.transform.a)
    delta_vv = vv_l - vv_e
    subsid   = (delta_vv < -0.15).astype('uint8')
    prof.update(count=1,dtype='uint8',compress='lzw')
    with rasterio.open(str(SUBSIDENCE),'w',**prof) as dst: dst.write(subsid[np.newaxis])
    print(f"Subsidence proxy: {subsid.sum()*res_m**2/10000:.1f} ha flagged")
    print("  Note: for operational use, combine with full DInSAR SLC time-series (NB07)")

In [ ]:
# Building extraction (SegFormer) + validate
BLDG_RAW=DATA_DIR/'buildings_raw.tif'
if S2LR.exists():
    with rasterio.open(str(S2LR)) as src: arr=src.read().astype('float32')
    print(f"SegFormer input: shape={arr.shape}  dtype={arr.dtype}  range=[{arr.min():.4f},{arr.max():.4f}]")
    assert arr.dtype=='float32' and arr.max()<=1.2
    model=get_model('segformer-b2',num_classes=2,in_channels=arr.shape[0],pretrained=True)
    inf=TiledInference(model,chip_size=512,overlap=64,blend_mode='gaussian')
    res=inf.infer(str(S2LR),str(BLDG_RAW))
    print(f"Buildings: {res.get('n_chips')} chips  {res.get('chips_per_second',0):.1f} chips/s")
    # DINOv3 urban character
    transform_fn=get_transform('dinov3_vitl16_sat')
    backbone=DINOv3Backbone('dinov3_vitl16_sat')
    emb=backbone.extract_embeddings(str(S2LR))
    print(f"DINOv3 urban embedding: {emb.shape}")

In [ ]:
# Urban change + postprocessing
URBA_CHANGE=DATA_DIR/'urban_change.tif'
if S2ER.exists() and S2LR.exists():
    cd=ChangeFormer(in_channels=len(S2_BANDS)); cd.build()
    res=cd.detect(str(S2ER),str(S2LR),output_path=str(URBA_CHANGE))
    print(f"Urban change 2021→2024: {res.get('change_pct',0):.2f}%")
    idx_dir=DATA_DIR/'indices'; idx_dir.mkdir(exist_ok=True)
    for yr,p in [('2021',S2ER),('2024',S2LR)]:
        ndbi=client.indices.ndbi(str(p),swir1_band=5,nir_band=4,output_path=str(idx_dir/f'ndbi_{yr}.tif'))
        a=rasterio.open(ndbi).read(1); print(f"  NDBI {yr}: built-up={( a>0.10).mean()*100:.1f}%")
for pred,lbl in [(BLDG_RAW,'buildings'),(SUBSIDENCE,'subsidence'),(URBA_CHANGE,'urban_change')]:
    if pred.exists():
        sieved=DATA_DIR/f'{lbl}_sieved.tif'; vec=DATA_DIR/f'{lbl}.geojson'
        client.postprocess.sieve_filter(str(pred),min_pixels=25,output_path=str(sieved))
        client.postprocess.vectorise(str(sieved),str(vec),target_class=1,min_area_m2=100.0)
        client.postprocess.to_cog(str(sieved),str(DATA_DIR/f'{lbl}_cog.tif'))
        stats=client.postprocess.class_statistics(str(sieved))
        with open(vec) as f: gj=json.load(f)
        print(f"{lbl}: {stats.get(1,{}).get('area_ha',0):.1f} ha  {len(gj['features'])} polygons")

---
## Fine-tuning: SegFormer-B2 Building Extraction on Jakarta Scene
Fine-tune decoder-only (frozen encoder) using NDBI-derived building pseudo-labels.


---
## Model Fine-tuning + PyGeoSpace Visualization

### Strategy: pseudo-label self-supervised fine-tuning

When labelled field data is unavailable, we use **spectral index pseudo-labels** as weak
supervision — the same physics-informed rules used by Prithvi's spectral predictor.
Fine-tuning the task head on domain-specific imagery consistently outperforms the
generic pre-trained baseline.

| Step | What happens | Time |
|---|---|---|
| 1. Chip extraction | Slice preprocessed scene into 256×256 tiles | < 1 min |
| 2. Pseudo-labeling | Spectral rules (NDVI/NDWI/NDBI) → weak labels | < 1 min |
| 3. Head fine-tuning | Freeze backbone, train task head only | ~5–15 min CPU |
| 4. Evaluate | mIoU, F1, accuracy on held-out 20% chips | < 1 min |
| 5. Compare | Pre-trained vs fine-tuned — PyGeoSpace split map | instant |
| 6. Export | ONNX + TorchScript for deployment | < 1 min |

**Backbone fine-tuning** (set `FINETUNE_BACKBONE=True`): unfreezes the last
2 transformer attention blocks with 10× lower LR — requires GPU, gives +2–8 mIoU.

### PyGeoSpace map patterns used in this notebook

```python
from pygeospace.folium import Map, RasterLayer, VectorLayer, SplitMapLayer
from pygeospace.integrations import geovision

m = Map(location=(lat, lon), zoom_start=11, basemap='carto-dark')
m.add_layer(RasterLayer(pred_array, bounds=BOUNDS, cmap='tab20'))
m.split_map(left_layer=RasterLayer(pretrained, ...), right_layer=RasterLayer(finetuned, ...))
m.add_legend({...}).add_colorbar(...).add_title(...).add_full_ui()
m.show()           # inline Jupyter
m.save("map.html") # standalone interactive
```


In [ ]:
# ── Dataset: Jakarta — building pseudo-labels from NDBI ────────────────────
if S2LR.exists():
    chips, sb = chips_from_raster(str(S2LR), CHIP_SIZE, max_chips=100)
    # Pseudo-label: NDBI > 0.10 = built-up | NDWI < 0 + NDVI < 0.1 = built
    labels_bld = []
    for chip in chips:
        G,R,NIR,SWIR1 = chip[1],chip[2],chip[3],chip[4]
        eps=1e-8
        ndbi=(SWIR1-NIR)/(SWIR1+NIR+eps); ndvi=(NIR-R)/(NIR+R+eps)
        labels_bld.append(((ndbi>0.10)&(ndvi<0.20)).astype(np.int64))
    labels_bld=np.stack(labels_bld)
    tr_bld, va_bld = make_loaders(chips, labels_bld, batch_size=4, split=TRAIN_VAL_SPLIT)
    print(f"Urban dataset: {len(chips)} chips  built-up={labels_bld.mean()*100:.1f}%")
else:
    tr_bld=va_bld=chips=labels_bld=None; print("S2 late ready not found")

In [ ]:
# ── PyGeoSpace: urban maps ────────────────────────────────────────────────────
BOUNDS=(JAKARTA_BBOX[0],JAKARTA_BBOX[1],JAKARTA_BBOX[2],JAKARTA_BBOX[3])
lat_c=(JAKARTA_BBOX[1]+JAKARTA_BBOX[3])/2; lon_c=(JAKARTA_BBOX[0]+JAKARTA_BBOX[2])/2

# Map 1: buildings split (baseline vs fine-tuned)
PRE_BLDG=DATA_DIR/'buildings_raw.tif'; FT_BLDG=DATA_DIR/'buildings_finetuned.tif'
if PRE_BLDG.exists() and FT_BLDG.exists():
    with rasterio.open(str(PRE_BLDG)) as s: bpre=s.read(1).astype('float32')
    with rasterio.open(str(FT_BLDG))  as s: bft=s.read(1).astype('float32')
    m1=PgsMap(location=(lat_c,lon_c), zoom_start=12, basemap='esri-satellite')
    m1.split_map(
        left_layer =RasterLayer(bpre,bounds=BOUNDS,cmap='Reds',opacity=0.75),
        right_layer=RasterLayer(bft, bounds=BOUNDS,cmap='Reds',opacity=0.75),
        left_label='Pre-trained SegFormer', right_label='Fine-tuned (Jakarta)')
    m1.add_title("Building Extraction — Jakarta",subtitle="Drag divider to compare")
    m1.add_full_ui()
    m1.save(str(DATA_DIR/'map_buildings_comparison.html')); m1.show()

# Map 2: subsidence risk + NDBI
m2=PgsMap(location=(lat_c,lon_c), zoom_start=11, basemap='carto-dark',
           title="Jakarta Subsidence Risk + Built-up Index")
SUBSID=DATA_DIR/'subsidence_proxy.tif'
if SUBSID.exists():
    with rasterio.open(str(SUBSID)) as src: arr=src.read(1).astype('float32')
    m2.add_layer(RasterLayer(arr,bounds=BOUNDS,cmap='YlOrRd',opacity=0.75))
idx_dir=DATA_DIR/'indices'
for yr,cmap in [('2021','Blues'),('2024','Oranges')]:
    ndbi_p=idx_dir/f'ndbi_{yr}.tif'
    if ndbi_p.exists():
        with rasterio.open(str(ndbi_p)) as src: arr=src.read(1).astype('float32')
        m2.add_layer(RasterLayer(arr,bounds=BOUNDS,cmap=cmap,opacity=0.5,vmin=-0.1,vmax=0.4))
m2.add_colorbar(-0.1,0.4,cmap='YlOrRd',caption='NDBI / Subsidence proxy')
m2.add_legend({'Subsidence risk':'#ff6600','Built-up 2021':'#0055ff','Built-up 2024':'#ff8800'})
for vec in ['subsidence_zones.geojson','buildings.geojson']:
    vp=DATA_DIR/vec
    if vp.exists():
        clr='#ff4400' if 'subsid' in vec else '#ffff00'
        m2.add_layer(VectorLayer(str(vp),fill_color=clr,fill_opacity=0.3,weight=1))
m2.add_title("Subsidence Risk — North Jakarta").add_scale()
m2.save(str(DATA_DIR/'map_subsidence.html')); m2.show()
print("NB06 maps saved")

---
## Model Fine-tuning with Registered Benchmark Datasets

**Primary registered datasets:** LEVIR-CD (637 high-quality building change pairs), WHU Building (8189 building segmentation 2-class), Inria Aerial (360 cities binary building), SpaceNet v1 (6940 samples 650×650)

**Why these?** LEVIR-CD is the most widely used benchmark for building change detection. Training SegFormer on LEVIR-CD + WHU Building gives strong building extraction that generalises across cities including Jakarta.

**3-tier strategy:** zero-shot baseline → benchmark-trained → domain fine-tuned


In [ ]:
# ── Registry: select and inspect relevant datasets ────────────────────────────
print("Registered datasets for this notebook domain:")
for name in ['LEVIR-CD', 'WHU Building', 'Inria Aerial', 'SpaceNet v1']:
    try:
        d=reg[name]
        print(f"\n  {name}")
        print(f"    Domain   : {d.domain}")
        print(f"    Modality : {d.modality}")
        print(f"    Samples  : {d.n_samples:,}")
        print(f"    Classes  : {d.n_classes}")
        print(f"    Size     : {d.volume_gb} GB")
        print(f"    Tasks    : {d.tasks}")
        print(f"    URL      : {d.download_url or '(request from authors / paper)'}")
        print(f"    Paper    : {d.paper_url}")
    except KeyError as e: print(f"  {name}: {e}")

# EarthNets similarity search
primary_ds = 'LEVIR-CD'
print(f"\nEarthNets datasets similar to {primary_ds}:")
for d in reg.similar_to(primary_ds, n=6):
    print(f"  {d.name:<28} {d.domain:<12} {d.n_samples:>8,}s  {d.n_classes}cls  {d.modality}")

# Top datasets for this task
task_key = reg.tasks()[0]  # browse available tasks
print("\nRegistry task list:", sorted(reg.tasks()))

In [ ]:
# ── Training pipeline: 3 tiers ───────────────────────────────────────────────
# TIER 0: Zero-shot inference with pre-trained model (already done in upper cells)
# TIER 1: Fine-tune on registered benchmark dataset
# TIER 2: Domain adapt on local study area scene

# DatasetLoader.download() gives the exact URL and size for any registered dataset
loader_inst = DatasetLoader(data_root='./datasets')
primary = 'LEVIR-CD'

print(f"TIER 1: Download + train on {primary}")
try:
    dl_path = loader_inst.download(primary, output_dir=f'./datasets/{primary}')
    print(f"  Data path: {dl_path}")
    print("  Run your data loading code here to build a DataLoader from the downloaded dataset")
    print("  Then: run_epochs(model, tr_loader, va_loader, epochs=50, criterion=DiceFocalLoss(), lr=1e-4)")
except ValueError as e:
    print(f"  {e}")
    print("  → Download from paper / project page, then load chips + labels into training loop")

# Example training loop structure (replace chips/labels with actual dataset)
print("\nTIER 2: Domain fine-tune on local study area scene (on top of Tier 1)")
print("  → After Tier 1 training: run_epochs(model, local_tr, local_va, epochs=10, lr=5e-5)")
print("  → 10x lower LR for domain adaptation (warm start from benchmark weights)")

# Show that we can query the best dataset for ANY custom task
print("\nRegistry query examples:")
print("  reg.top_for_task('segmentation', n=5)  → best segmentation datasets")
print("  reg.similar_to('xBD', n=5)             → EarthNets similarity ranking")
print("  reg.filter(domain='disaster', modality='sar', task='segmentation')")
print("  reg.search('oil spill')                → keyword search across 503 datasets")

# Show full comparison table with registered dataset context
print("\nModel comparison context:")
print(f"  Registered primary dataset: {primary}")
d = reg[primary]
print(f"    Training samples: {d.n_samples:,}")
print(f"    Classes: {d.n_classes}")
print(f"    Modality: {d.modality}")
print("  → Benchmark-trained model should outperform zero-shot by 5-20 mIoU")
print("  → Domain fine-tuned (Tier 2) typically adds another 3-8 mIoU")

In [ ]:
# ── PyGeoSpace: results maps ─────────────────────────────────────────────────
# Use whichever preprocessed path is available in this notebook
ready_candidates = [v for v in dir() if 'READY' in v and pathlib.Path(str(eval(v))).exists() if not v.startswith('__')]
print("Available preprocessed scenes:", ready_candidates)

# Create a representative map for this notebook's domain
# (replace with actual prediction paths from fine-tuned model inference)
# PyGeoSpace imported in Setup cell (Cell 0)

# Dataset ecosystem analysis
analyzer = DatasetAnalyzer(reg)
print("\nDataset volume trend across years:")
analyzer.volume_trend()

print("\nRegistered benchmark datasets available for this notebook:")
print("  Primary  : LEVIR-CD → fine-tune Tier 1")
print("  Secondary: WHU Building, Inria Aerial, SpaceNet v1 → augment / cross-validate")
print("\nTo load into training:")
print("  1. DatasetLoader().download('dataset_name', output_dir='./data')")
print("  2. Load chips + labels from downloaded directory")
print("  3. make_loaders(chips, labels) → tr_loader, va_loader")
print("  4. run_epochs(model, tr, va, epochs=50, criterion=DiceFocalLoss())")
print("  5. compare3(pred_zero, pred_bench, pred_ft, labels)")

In [ ]:
# ── Export + complete registry report ────────────────────────────────────────
print("Complete registry report for this notebook:")
print(f"  {'Dataset':<24} {'Modality':<14} {'Samples':>10} {'Classes':>8} {'GB':>6}")
print("  "+"-"*68)
for name in ['LEVIR-CD', 'WHU Building', 'Inria Aerial', 'SpaceNet v1']:
    try:
        d=reg[name]
        print(f"  {d.name:<24} {d.modality:<14} {d.n_samples:>10,} {d.n_classes:>8} {d.volume_gb:>6.0f}")
    except: pass

# Summary statistics
total_samples=sum(reg[n].n_samples for n in ['LEVIR-CD', 'WHU Building', 'Inria Aerial', 'SpaceNet v1'] if n in reg._by_name)
total_gb=sum(reg[n].volume_gb for n in ['LEVIR-CD', 'WHU Building', 'Inria Aerial', 'SpaceNet v1'] if n in reg._by_name)
print(f"\n  Total training data available: {total_samples:,} samples  {total_gb:.0f} GB")
print("  → All 503 registered datasets browseable via DatasetRegistry()")
print("  → Use reg.similar_to(), reg.top_for_task(), reg.filter() for intelligent selection")
print("  → DatasetLoader().info(name) shows download URL for any dataset")